# RAGAS-based RAG Evaluation Notebook
This notebook evaluates Retrieval Quality and Tool Use Accuracy using the `unittest` framework.

In [21]:
from __future__ import annotations

# --- Standard Library Imports ---
import asyncio
import json
import os
import time
import unittest
from pathlib import Path
from typing import Any

# --- Third-Party Data & Env Imports ---
import pandas as pd
from dotenv import load_dotenv

# --- OpenAI & ChromaDB Imports ---
import chromadb
from chromadb.utils import embedding_functions
from openai import AsyncOpenAI

# --- RAGAS Imports ---
from ragas.embeddings.base import embedding_factory
from ragas.llms import llm_factory
from ragas.metrics.collections import AnswerRelevancy, Faithfulness, ToolCallAccuracy

# RAGAS specific message classes to prevent internal type-checking failures
from ragas.messages import AIMessage as RagasAIMessage
from ragas.messages import HumanMessage as RagasHumanMessage
from ragas.messages import ToolCall as RagasToolCall


# --- ENVIRONMENT SETUP ---
def _load_dotenv():
    """
    Loads environment variables from .env file in the current, parent, or tests directory.
    """
    _cwd = Path.cwd()
    env_paths = [
        _cwd / ".env",
        _cwd.parent / ".env",
        _cwd / "tests" / ".env"
    ]
    for _env in env_paths:
        if _env.is_file():
            load_dotenv(_env)
            break
    else:
        load_dotenv()

_load_dotenv()


# --- CONFIGURATION & THRESHOLDS ---

# File paths
GOLDEN_DATASET_PATH = Path("golden_dataset.json")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Evaluation thresholds
FAITHFULNESS_THRESHOLD = 0.4
ANSWER_RELEVANCY_THRESHOLD = 0.60
TOOL_ACCURACY_THRESHOLD = 0.80

# Performance and Truncation Constants
MAX_AVERAGE_RESPONSE_TIME = 12.0
# Set separate percentages for the answer and the context (e.g., 0.30 = 30%)
ANSWER_TRUNCATION_PERCENTAGE = 0.30
CONTEXT_TRUNCATION_PERCENTAGE = 0.30

# RAGAS and OpenAI Model configs
RAGAS_LLM_MODEL = os.getenv("RAGAS_LLM_MODEL", "gpt-5.4-mini")
RAGAS_LLM_TEMPERATURE = 0.0
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# Embedding Function for ChromaDB
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name="text-embedding-3-small"
)

In [3]:
# --- HELPERS ---

def _load_golden_dataset() -> list[dict]:
    if not GOLDEN_DATASET_PATH.exists():
        return []
    with open(GOLDEN_DATASET_PATH) as f:
        return json.load(f)

def _get_ragas_llm():
    from openai import OpenAI
    from ragas.llms import llm_factory

    client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else OpenAI()
    return llm_factory(
        RAGAS_LLM_MODEL,
        client=client,
        temperature=RAGAS_LLM_TEMPERATURE,
    )


def _get_ragas_embeddings():
    from openai import OpenAI
    from ragas.embeddings import OpenAIEmbeddings

    client = OpenAI(api_key=OPENAI_API_KEY) if OPENAI_API_KEY else OpenAI()
    return OpenAIEmbeddings(client=client, model="text-embedding-3-small")

def _save_to_excel(data: list[dict] | pd.DataFrame, test_name: str):
    date_str = time.strftime("%Y%m%d")
    filename = RESULTS_DIR / f"{test_name}_{date_str}.xlsx"
    df = pd.DataFrame(data) if isinstance(data, list) else data
    df = df.loc[:, ~df.columns.duplicated()]
    df.to_excel(filename, index=False)
    print(f"[INFO] Results saved to {filename}")

In [85]:
class BaseRetrievalTest(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.chroma_url = os.getenv("CHROMA_URL")
        if not cls.chroma_url:
            raise unittest.SkipTest("CHROMA_URL not set")
        
        cls.client = chromadb.HttpClient(
            host=cls.chroma_url,
            headers={
                "CF-Access-Client-Id": os.getenv("CF-ACCESS-CLIENT-ID", ""),
                "CF-Access-Client-Secret": os.getenv("CF-ACCESS-CLIENT-SECRET", ""),
            },
        )
        cls.dataset = _load_golden_dataset()
        if not cls.dataset:
            raise unittest.SkipTest("Golden dataset is empty or missing")

    def run_hit_rate_test(self, target_collection_prefix, model_sufix, ef=None):
        subset = [d for d in self.dataset if d["expected_collection"] == target_collection_prefix]
        hits, results_detail = 0, []
        
        for case in subset:
            try:
                collection = self.client.get_collection(name=f'{target_collection_prefix}_{model_sufix}', embedding_function=ef)
                results = collection.query(
                    query_texts=[case["question"]], 
                    n_results=5, 
                    where={"ticker": case["ticker"]} if case.get("ticker") else None
                )
                print(results)
                docs = results.get("documents", [[]])[0]
                hit = any(any(kw.lower() in doc.lower() for kw in case.get("expected_keywords", [])) for doc in docs)
                if hit: hits += 1
                results_detail.append({"id": case["id"], "question": case["question"], "hit": hit})
            except Exception as e:
                results_detail.append({"id": case["id"], "error": str(e)})
                
        _save_to_excel(results_detail, f"hit_rate_{target_collection_prefix}_{model_sufix}")
        self.assertGreaterEqual(hits / len(subset), 0.70)

    def run_similarity_scores(self, collection_map, output_prefix, ef=None):
        subset = [d for d in self.dataset if d.get("expected_collection")]
        results_detail, low_score_count = [], 0
        
        for case in subset:
            physical_col = collection_map.get(case["expected_collection"])
            try:
                collection = self.client.get_collection(name=physical_col, embedding_function=ef)        
                results = collection.query(
                    query_texts=[case["question"]], 
                    n_results=1, 
                    where={"ticker": case["ticker"]} if case.get("ticker") else None,
                    include=["distances"]
                )
                print(f"The result is : {results}")
                dist = results.get("distances", [[]])[0][0] if results.get("distances") else 1.0
                similarity = 1 - dist
                results_detail.append({"id": case["id"], "similarity": similarity, "passed": similarity >= 0.5})
                if similarity < 0.5: low_score_count += 1
            except Exception as e:
                results_detail.append({"id": case["id"], "error": str(e)})
                
        _save_to_excel(results_detail, f"similarity_{output_prefix}")
        self.assertEqual(low_score_count, 0)

In [86]:
class TestRetrievalQualityChroma(BaseRetrievalTest):
    def test_sec_filings_hit_rate(self):
        self.run_hit_rate_test("sec_filings","chroma")
    def test_news_hit_rate(self):
        self.run_hit_rate_test("news","chroma")
    def test_similarity_scores(self):
        self.run_similarity_scores({"sec_filings": "sec_filings_chroma", "news": "news_chroma"}, "chroma")

# Run Chroma Tests Immediately
suite = unittest.TestLoader().loadTestsFromTestCase(TestRetrievalQualityChroma)
unittest.TextTestRunner(verbosity=2).run(suite)

test_news_hit_rate (__main__.TestRetrievalQualityChroma.test_news_hit_rate) ... 

{'ids': [['TSLA-cb66e03c-fd11-3041-8418-7b680cc068e6-9', 'TSLA-2375c50f-3f3f-3dbe-8627-5b547e79d510-3', 'TSLA-2375c50f-3f3f-3dbe-8627-5b547e79d510-5', 'TSLA-cb66e03c-fd11-3041-8418-7b680cc068e6-0', 'TSLA-2375c50f-3f3f-3dbe-8627-5b547e79d510-0']], 'distances': [[0.36419398, 0.43007928, 0.43266028, 0.45270962, 0.4558211]], 'embeddings': None, 'metadatas': [[{'published': 1777212002, 'ticker': 'TSLA', 'link': 'https://finance.yahoo.com/m/cb66e03c-fd11-3041-8418-7b680cc068e6/better-later-than-never%3F.html', 'sentiment_negative': 0.2211, 'chunk_count': 13, 'sentiment_positive': 0.7598, 'publisher': 'Barchart', 'sentiment_score': 0.5386, 'quality': 'high', 'sentiment_label': 'positive', 'sentiment_neutral': 0.0191, 'title': 'Better Later Than Never? Tesla Stock Hangs in the Crosshairs as Cybercab Finally Enters Production', 'original_uuid': 'cb66e03c-fd11-3041-8418-7b680cc068e6', 'chunk_index': 9}, {'sentiment_positive': 0.2826, 'sentiment_label': 'neutral', 'sentiment_neutral': 0.6926, 'se

ok
test_sec_filings_hit_rate (__main__.TestRetrievalQualityChroma.test_sec_filings_hit_rate) ... 

{'ids': [['GOOGL-decf9cfc-f34e-3952-b813-80b2bc232edb-5', 'GOOGL-a6ecc093-d7e3-3aac-be54-0614945695a7-8', 'GOOGL-a6ecc093-d7e3-3aac-be54-0614945695a7-6', 'GOOGL-decf9cfc-f34e-3952-b813-80b2bc232edb-0', 'GOOGL-b16aacaf-99ff-31f9-b81f-b32834b77e3b-3']], 'distances': [[0.41176945, 0.4360745, 0.4364156, 0.45212382, 0.4758019]], 'embeddings': None, 'metadatas': [[{'sentiment_positive': 0.3836, 'chunk_index': 5, 'published': 1777223098, 'ticker': 'GOOGL', 'sentiment_score': 0.3096, 'link': 'https://finance.yahoo.com/markets/stocks/articles/bull-case-alphabet-googl-could-170458027.html', 'publisher': 'Simply Wall St.', 'sentiment_label': 'neutral', 'sentiment_neutral': 0.5424, 'chunk_count': 6, 'sentiment_negative': 0.074, 'quality': 'high', 'original_uuid': 'decf9cfc-f34e-3952-b813-80b2bc232edb', 'title': 'The Bull Case For Alphabet (GOOGL) Could Change Following Massive AI Bet On Anthropic Partnership'}, {'sentiment_score': 0.4812, 'chunk_index': 8, 'sentiment_label': 'positive', 'link': 'h

ok
test_similarity_scores (__main__.TestRetrievalQualityChroma.test_similarity_scores) ... 

{'ids': [['e121537f-6d46-52ec-aab7-befb9e519150', '9b960bd0-ac57-5340-8fae-a81c039f6926', '57311865-0ae1-5dd3-9930-091a28dcf91c', '86c47381-9552-5c52-aed5-74e603266426', '6307c5c9-42bf-50a2-aba2-90f6a779a909']], 'distances': [[0.38550264, 0.38900596, 0.4231134, 0.44317943, 0.45368016]], 'embeddings': None, 'metadatas': [[{'ticker': 'MSFT', 'source': 'Microsoft Corporation 10-K', 'id': 'e121537f-6d46-52ec-aab7-befb9e519150', 'company': 'Microsoft Corporation'}, {'ticker': 'MSFT', 'source': 'Microsoft Corporation 10-K', 'company': 'Microsoft Corporation', 'id': '9b960bd0-ac57-5340-8fae-a81c039f6926'}, {'source': 'Microsoft Corporation 10-K', 'ticker': 'MSFT', 'id': '57311865-0ae1-5dd3-9930-091a28dcf91c', 'company': 'Microsoft Corporation'}, {'company': 'Microsoft Corporation', 'id': '86c47381-9552-5c52-aed5-74e603266426', 'ticker': 'MSFT', 'source': 'Microsoft Corporation 10-K'}, {'id': '6307c5c9-42bf-50a2-aba2-90f6a779a909', 'company': 'Microsoft Corporation', 'source': 'Microsoft Corpo

FAIL

FAIL: test_similarity_scores (__main__.TestRetrievalQualityChroma.test_similarity_scores)
----------------------------------------------------------------------
Traceback (most recent call last):
  File "/var/folders/2f/4qv4v6_s6714t7lb2hc6jtqh0000gn/T/ipykernel_14287/4258937053.py", line 7, in test_similarity_scores
    self.run_similarity_scores({"sec_filings": "sec_filings_chroma", "news": "news_chroma"}, "chroma")
    ~~~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/2f/4qv4v6_s6714t7lb2hc6jtqh0000gn/T/ipykernel_14287/564112911.py", line 65, in run_similarity_scores
    self.assertEqual(low_score_count, 0)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^
AssertionError: 2 != 0

----------------------------------------------------------------------
Ran 3 tests in 23.537s

FAILED (failures=1)


The result is : {'ids': [['GOOGL-decf9cfc-f34e-3952-b813-80b2bc232edb-5']], 'distances': [[0.41176945]], 'embeddings': None, 'metadatas': None, 'documents': None, 'uris': None, 'data': None, 'included': ['distances']}
[INFO] Results saved to results/similarity_chroma_20260427.xlsx


<unittest.runner.TextTestResult run=3 errors=0 failures=1>

In [87]:
class TestRetrievalQualityOpenAI(BaseRetrievalTest):
    def test_sec_filings_hit_rate(self):
        self.run_hit_rate_test("sec_filings", "openai", ef=openai_ef)
    def test_news_hit_rate(self):
        self.run_hit_rate_test("news", "openai", ef=openai_ef)
    def test_similarity_scores(self):
        self.run_similarity_scores({"sec_filings": "sec_filings_openai", "news": "news_openai"}, "openai", ef=openai_ef)

# Run OpenAI Tests Immediately
suite = unittest.TestLoader().loadTestsFromTestCase(TestRetrievalQualityOpenAI)
unittest.TextTestRunner(verbosity=2).run(suite)

test_news_hit_rate (__main__.TestRetrievalQualityOpenAI.test_news_hit_rate) ... 

{'ids': [['TSLA-cb66e03c-fd11-3041-8418-7b680cc068e6-0', 'TSLA-cb66e03c-fd11-3041-8418-7b680cc068e6-10', 'TSLA-2375c50f-3f3f-3dbe-8627-5b547e79d510-4', 'TSLA-cb66e03c-fd11-3041-8418-7b680cc068e6-9', 'TSLA-e222e246-88e5-391b-9332-ed8446f82a9c-2']], 'distances': [[0.3633901, 0.3870296, 0.39664602, 0.39849788, 0.3988092]], 'embeddings': None, 'metadatas': [[{'title': 'Better Later Than Never? Tesla Stock Hangs in the Crosshairs as Cybercab Finally Enters Production', 'chunk_index': 0, 'quality': 'high', 'link': 'https://finance.yahoo.com/m/cb66e03c-fd11-3041-8418-7b680cc068e6/better-later-than-never%3F.html', 'sentiment_positive': 0.7598, 'sentiment_neutral': 0.0191, 'sentiment_negative': 0.2211, 'ticker': 'TSLA', 'published': 1777212002, 'publisher': 'Barchart', 'chunk_count': 13, 'sentiment_label': 'positive', 'sentiment_score': 0.5386, 'original_uuid': 'cb66e03c-fd11-3041-8418-7b680cc068e6'}, {'sentiment_neutral': 0.0191, 'publisher': 'Barchart', 'chunk_index': 10, 'chunk_count': 13, '

ok
test_sec_filings_hit_rate (__main__.TestRetrievalQualityOpenAI.test_sec_filings_hit_rate) ... 

{'ids': [['GOOGL-a6ecc093-d7e3-3aac-be54-0614945695a7-8', 'GOOGL-decf9cfc-f34e-3952-b813-80b2bc232edb-5', 'GOOGL-decf9cfc-f34e-3952-b813-80b2bc232edb-4', 'GOOGL-b16aacaf-99ff-31f9-b81f-b32834b77e3b-10', 'GOOGL-a6ecc093-d7e3-3aac-be54-0614945695a7-6']], 'distances': [[0.43728763, 0.43866754, 0.44132966, 0.48820198, 0.49561667]], 'embeddings': None, 'metadatas': [[{'sentiment_label': 'positive', 'sentiment_score': 0.4812, 'title': 'Mega-Cap Earnings, FOMC and Other Key Things to Watch this Week', 'quality': 'high', 'chunk_count': 19, 'original_uuid': 'a6ecc093-d7e3-3aac-be54-0614945695a7', 'sentiment_negative': 0.0332, 'publisher': 'Barchart', 'sentiment_neutral': 0.4524, 'link': 'https://finance.yahoo.com/m/a6ecc093-d7e3-3aac-be54-0614945695a7/mega-cap-earnings%2C-fomc-and.html', 'ticker': 'GOOGL', 'sentiment_positive': 0.5144, 'chunk_index': 8, 'published': 1777222802}, {'sentiment_label': 'neutral', 'published': 1777223098, 'sentiment_neutral': 0.5424, 'ticker': 'GOOGL', 'quality': 'h

ok
test_similarity_scores (__main__.TestRetrievalQualityOpenAI.test_similarity_scores) ... 

{'ids': [['9b960bd0-ac57-5340-8fae-a81c039f6926', 'ea888979-f706-5174-9786-41ff5075f42d', 'fe02c5c4-c4ea-5ffb-b887-5904daf9c018', '57311865-0ae1-5dd3-9930-091a28dcf91c', '8d981f47-5903-572b-a8e9-778b5e5962f2']], 'distances': [[0.41482288, 0.42649525, 0.45132607, 0.4707927, 0.4736573]], 'embeddings': None, 'metadatas': [[{'id': '9b960bd0-ac57-5340-8fae-a81c039f6926', 'source': 'Microsoft Corporation 10-K', 'company': 'Microsoft Corporation', 'ticker': 'MSFT'}, {'id': 'ea888979-f706-5174-9786-41ff5075f42d', 'company': 'Microsoft Corporation', 'ticker': 'MSFT', 'source': 'Microsoft Corporation 10-K'}, {'source': 'Microsoft Corporation 10-K', 'ticker': 'MSFT', 'company': 'Microsoft Corporation', 'id': 'fe02c5c4-c4ea-5ffb-b887-5904daf9c018'}, {'source': 'Microsoft Corporation 10-K', 'company': 'Microsoft Corporation', 'ticker': 'MSFT', 'id': '57311865-0ae1-5dd3-9930-091a28dcf91c'}, {'company': 'Microsoft Corporation', 'source': 'Microsoft Corporation 10-K', 'ticker': 'MSFT', 'id': '8d981f47

ok

----------------------------------------------------------------------
Ran 3 tests in 36.898s

OK


The result is : {'ids': [['GOOGL-a6ecc093-d7e3-3aac-be54-0614945695a7-8']], 'distances': [[0.43728763]], 'embeddings': None, 'metadatas': None, 'documents': None, 'uris': None, 'data': None, 'included': ['distances']}
[INFO] Results saved to results/similarity_openai_20260427.xlsx


<unittest.runner.TextTestResult run=3 errors=0 failures=0>

In [ ]:
class TestFaithfulness(unittest.TestCase): 
    
    @classmethod
    def setUpClass(cls):
        load_dotenv()
        cls.backend_url = os.getenv("BACKEND_URL", "http://localhost:8001")
        
        # Load Dataset
        try:
            cls.dataset = _load_golden_dataset() 
            print(f"✅ Dataset loaded: {len(cls.dataset)} cases found.")
        except NameError:
            cls.dataset = []
            print("⚠️ Warning: _load_golden_dataset helper not found.")

        # 3. Modern LLM Initialization (Ragas 1.0)
        # We use AsyncOpenAI to ensure compatibility with Ragas' internal async calls
        client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        cls.ragas_llm = llm_factory("gpt-5.4-mini", client=client)

    def _call_agent(self, question: str, ticker: str | None = None) -> dict:
        """Standard sync httpx call (following your worked example)"""
        payload = {"message": f"[Ticker: {ticker}] {question}" if ticker else question}
        with httpx.Client() as client:
            resp = client.post(f"{self.backend_url}/api/v1/agent/chat", json=payload, timeout=60)
            resp.raise_for_status()
            data = resp.json()
        
        # Extract context outputs from tool_calls
        tool_calls = data.get("tool_calls", [])
        raw_outputs = [tc["output"] for tc in tool_calls if "output" in tc]
        
        clean_contexts = []
        for ctx in raw_outputs:
            if isinstance(ctx, str): clean_contexts.append(ctx)
            elif isinstance(ctx, list):
                for item in ctx:
                    if isinstance(item, dict): clean_contexts.append(item.get("text") or str(item))
                    else: clean_contexts.append(str(item))
        
        return {
            "answer": data.get("answer", ""), 
            "contexts": clean_contexts, 
            "raw_tool_calls": tool_calls 
        }

    async def _run_ragas_and_export_async(self, rows, metric, test_name):
        """Async helper to batch process evaluation rows"""
        if not rows: 
            return pd.DataFrame()
            
        results_data = []
        print(f"\n--- Starting RAGAS Scoring for {test_name} ---")
        for row in rows:
            eval_kwargs = {
                "user_input": row["question"], 
                "response": row["answer"],
                "retrieved_contexts": row["contexts"]
            }
                
            # Perform asynchronous scoring
            result = await metric.ascore(**eval_kwargs)
            score_value = result.value if hasattr(result, 'value') else float(result)
            
            row_copy = row.copy()
            row_copy[metric.name] = round(score_value, 4)
            results_data.append(row_copy)
            
            # Print individual result for tracking
            print(f"📊 [Score Calculated] Question: {row['question'][:40]}... | Result: {score_value:.4f}")
            
        df = pd.DataFrame(results_data)
        try: _save_to_excel(df, test_name) 
        except NameError: pass
        return df

    # --- UPDATED TEST: FAITHFULNESS ---
    def test_faithfulness(self):
        async def _run_test_logic():
            scorer = Faithfulness(llm=self.ragas_llm)
            subset = [d for d in self.dataset if d.get("expected_collection")]
            rows = []
            
            print(f"\n🚀 [FAITHFULNESS] Starting evaluation on {min(len(subset), 20)} cases...")
            print(f"⚙️ Truncation Config: Answer @ {ANSWER_TRUNCATION_PERCENTAGE*100}% | Context @ {CONTEXT_TRUNCATION_PERCENTAGE*100}%")

            for case in subset[:20]:
                case_id = case.get('id', 'unknown')
                try:
                    print(f"🔎 [{case_id}] Calling agent...")
                    res = self._call_agent(case["question"], case.get("ticker"))
                    
                    # 1. Dynamic Answer Truncation
                    ans = res.get("answer", "")
                    ans_len = len(ans)
                    ans_limit = int(ans_len * ANSWER_TRUNCATION_PERCENTAGE)
                    safe_answer = ans[:ans_limit] + "..." if ans_len > 0 else ""
                        
                    # 2. Dynamic Context Truncation
                    raw_contexts = res.get("contexts") or [case.get("ground_truth", "")]
                    safe_contexts = []
                    
                    for c in raw_contexts:
                        try:
                            # Handle JSON-structured tool output
                            parsed = json.loads(c) if isinstance(c, str) else c
                            # Sort by relevance if metadata is available
                            data_items = sorted(parsed.get("data", []), key=lambda x: x.get("similarity_score", 0.0), reverse=True)
                            combined_text = " ".join([item.get("document", "") for item in data_items])
                            
                            # Truncate relative to the size of this specific combined text
                            limit = int(len(combined_text) * CONTEXT_TRUNCATION_PERCENTAGE)
                            safe_contexts.append(combined_text[:limit] + "...")
                            
                        except (json.JSONDecodeError, AttributeError, TypeError):
                            # Fallback for plain text context
                            c_str = str(c)
                            limit = int(len(c_str) * CONTEXT_TRUNCATION_PERCENTAGE)
                            safe_contexts.append(c_str[:limit] + "...")

                    rows.append({
                        "question": case["question"], 
                        "answer": safe_answer, 
                        "contexts": safe_contexts,
                        "ground_truth": case.get("ground_truth", "")
                    })
                    print(f"✅ [{case_id}] Agent call successful. (Answer truncated to {ans_limit} chars)")

                except Exception as e: 
                    print(f"❌ [{case_id}] Agent call failed: {e}")
                    continue
            
            # Execute Ragas scoring
            df_results = await self._run_ragas_and_export_async(rows, scorer, "ragas_faithfulness")
            
            mean_score = df_results[scorer.name].mean() if not df_results.empty else 0.0
            print(f"📈 [FAITHFULNESS] Final Mean Score: {mean_score:.4f}")
            return mean_score

        # Bridging Sync test to Async logic using existing loop
        loop = asyncio.get_event_loop()
        score = loop.run_until_complete(_run_test_logic())
        
        self.assertGreaterEqual(score, FAITHFULNESS_THRESHOLD)

# 4. Execution Block for Notebook
if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestFaithfulness)
    unittest.TextTestRunner(verbosity=2).run(suite)

✅ Dataset loaded: 52 cases found.


test_faithfulness (__main__.TestFaithfulness.test_faithfulness) ... 


🚀 [FAITHFULNESS] Starting evaluation on 16 cases...
⚙️ Truncation Config: Answer @ 30.0% | Context @ 30.0%
🔎 [gd-001] Calling agent...
✅ [gd-001] Agent call successful. (Answer truncated to 484 chars)
🔎 [gd-003] Calling agent...
✅ [gd-003] Agent call successful. (Answer truncated to 708 chars)
🔎 [gd-005] Calling agent...
✅ [gd-005] Agent call successful. (Answer truncated to 622 chars)
🔎 [gd-007] Calling agent...
✅ [gd-007] Agent call successful. (Answer truncated to 180 chars)
🔎 [gd-009] Calling agent...
✅ [gd-009] Agent call successful. (Answer truncated to 876 chars)
🔎 [gd-010] Calling agent...
✅ [gd-010] Agent call successful. (Answer truncated to 800 chars)
🔎 [gd-012] Calling agent...
✅ [gd-012] Agent call successful. (Answer truncated to 513 chars)
🔎 [gd-034] Calling agent...
✅ [gd-034] Agent call successful. (Answer truncated to 275 chars)
🔎 [gd-035] Calling agent...
✅ [gd-035] Agent call successful. (Answer truncated to 488 chars)
🔎 [gd-036] Calling agent...
✅ [gd-036] Agent c

ok

----------------------------------------------------------------------
Ran 1 test in 321.609s

OK


📊 [Score Calculated] Question: What are analysts saying about Google's ... | Result: 0.0000
[INFO] Results saved to results/ragas_faithfulness_20260428.xlsx
📈 [FAITHFULNESS] Final Mean Score: 0.4326


In [23]:
class TestAnswerRelevancy(unittest.TestCase): 
    
    @classmethod
    def setUpClass(cls):
        load_dotenv()
        cls.backend_url = os.getenv("BACKEND_URL", "http://localhost:8001")
        
        # Load Dataset
        try:
            cls.dataset = _load_golden_dataset() 
            print(f"✅ Dataset loaded: {len(cls.dataset)} cases found.")
        except NameError:
            cls.dataset = []
            print("⚠️ Warning: _load_golden_dataset helper not found.")

        # 3. Modern LLM & Embeddings Initialization
        client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        cls.ragas_llm = llm_factory("gpt-5.4-mini", client=client)
        cls.ragas_embeddings = embedding_factory("openai", model="text-embedding-3-small", client=client)

    def _call_agent(self, question: str, ticker: str | None = None) -> dict:
        """Standard sync httpx call (following your worked example)"""
        payload = {"message": f"[Ticker: {ticker}] {question}" if ticker else question}
        with httpx.Client() as client:
            resp = client.post(f"{self.backend_url}/api/v1/agent/chat", json=payload, timeout=60)
            resp.raise_for_status()
            data = resp.json()
        
        return {
            "answer": data.get("answer", ""), 
            "raw_tool_calls": data.get("tool_calls", []) 
        }

    async def _run_ragas_and_export_async(self, rows, metric, test_name):
        """Async helper to batch process evaluation rows"""
        if not rows: 
            return pd.DataFrame()
            
        results_data = []
        print(f"\n--- Starting RAGAS Scoring for {test_name} ---")
        for row in rows:
            eval_kwargs = {
                "user_input": row["question"], 
                "response": row["answer"]
            }
                
            # Answer Relevancy typically doesn't strictly require contexts, 
            # but uses embeddings to compare input vs response.
            result = await metric.ascore(**eval_kwargs)
            score_value = result.value if hasattr(result, 'value') else float(result)
            
            row_copy = row.copy()
            row_copy[metric.name] = round(score_value, 4)
            results_data.append(row_copy)
            
            # Print individual result for tracking
            print(f"📊 [Score Calculated] Question: {row['question'][:40]}... | Result: {score_value:.4f}")
            
        df = pd.DataFrame(results_data)
        try: _save_to_excel(df, test_name) 
        except NameError: pass
        return df

    # --- UPDATED TEST: ANSWER RELEVANCY ---
    def test_answer_relevancy(self):
        async def _run_test_logic():
            # Initialize scorer with embeddings
            scorer = AnswerRelevancy(llm=self.ragas_llm, embeddings=self.ragas_embeddings)
            subset = self.dataset[:20] # Or whatever subset logic you prefer
            rows = []
            
            print(f"\n🚀 [RELEVANCY] Starting evaluation on {min(len(subset), 20)} cases...")
            print(f"⚙️ Truncation Config: Answer @ {ANSWER_TRUNCATION_PERCENTAGE*100}%")

            for case in subset:
                case_id = case.get('id', 'unknown')
                try:
                    print(f"🔎 [{case_id}] Calling agent...")
                    res = self._call_agent(case["question"], case.get("ticker"))
                    
                    # 1. Dynamic Answer Truncation (Percentage-based)
                    ans = res.get("answer", "")
                    ans_len = len(ans)
                    ans_limit = int(ans_len * ANSWER_TRUNCATION_PERCENTAGE)
                    safe_answer = ans[:ans_limit] + "..." if ans_len > 0 else ""
                        
                    rows.append({
                        "question": case["question"], 
                        "answer": safe_answer, 
                        "ground_truth": case.get("ground_truth", "")
                    })
                    print(f"✅ [{case_id}] Agent call successful. (Answer truncated to {ans_limit} chars)")

                except Exception as e: 
                    print(f"❌ [{case_id}] Agent call failed: {e}")
                    continue
            
            # Execute Ragas scoring
            df_results = await self._run_ragas_and_export_async(rows, scorer, "ragas_relevancy")
            
            mean_score = df_results[scorer.name].mean() if not df_results.empty else 0.0
            print(f"📈 [RELEVANCY] Final Mean Score: {mean_score:.4f}")
            return mean_score

        # Bridging Sync test to Async logic using existing loop
        loop = asyncio.get_event_loop()
        score = loop.run_until_complete(_run_test_logic())
        
        self.assertGreaterEqual(score, ANSWER_RELEVANCY_THRESHOLD)

# 4. Execution Block
if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestAnswerRelevancy)
    unittest.TextTestRunner(verbosity=2).run(suite)

✅ Dataset loaded: 52 cases found.


test_answer_relevancy (__main__.TestAnswerRelevancy.test_answer_relevancy) ... 


🚀 [RELEVANCY] Starting evaluation on 20 cases...
⚙️ Truncation Config: Answer @ 30.0%
🔎 [gd-001] Calling agent...
✅ [gd-001] Agent call successful. (Answer truncated to 852 chars)
🔎 [gd-002] Calling agent...
✅ [gd-002] Agent call successful. (Answer truncated to 30 chars)
🔎 [gd-003] Calling agent...
✅ [gd-003] Agent call successful. (Answer truncated to 687 chars)
🔎 [gd-004] Calling agent...
✅ [gd-004] Agent call successful. (Answer truncated to 57 chars)
🔎 [gd-005] Calling agent...
✅ [gd-005] Agent call successful. (Answer truncated to 679 chars)
🔎 [gd-006] Calling agent...
✅ [gd-006] Agent call successful. (Answer truncated to 756 chars)
🔎 [gd-007] Calling agent...
✅ [gd-007] Agent call successful. (Answer truncated to 205 chars)
🔎 [gd-008] Calling agent...
✅ [gd-008] Agent call successful. (Answer truncated to 114 chars)
🔎 [gd-009] Calling agent...
✅ [gd-009] Agent call successful. (Answer truncated to 756 chars)
🔎 [gd-010] Calling agent...
✅ [gd-010] Agent call successful. (Answer

ok

----------------------------------------------------------------------
Ran 1 test in 251.864s

OK


📊 [Score Calculated] Question: Give me Amazon's price chart for the las... | Result: 0.8352
[INFO] Results saved to results/ragas_relevancy_20260428.xlsx
📈 [RELEVANCY] Final Mean Score: 0.7333


In [24]:
class TestToolUseAccuracy(unittest.TestCase): 
    
    @classmethod
    def setUpClass(cls):
        load_dotenv()
        cls.backend_url = os.getenv("BACKEND_URL", "http://localhost:8001")
        
        # Load Dataset
        try:
            cls.dataset = _load_golden_dataset() 
            print(f"✅ Dataset loaded: {len(cls.dataset)} cases found.")
        except NameError:
            cls.dataset = []
            print("⚠️ Warning: _load_golden_dataset helper not found.")

        # 3. Modern LLM Initialization
        client = AsyncOpenAI(api_key=os.getenv("OPENAI_API_KEY"))
        cls.ragas_llm = llm_factory("gpt-5.4-mini", client=client)

    def _call_agent(self, question: str, ticker: str | None = None) -> dict:
        """Standard sync httpx call (following your worked example)"""
        payload = {"message": f"[Ticker: {ticker}] {question}" if ticker else question}
        with httpx.Client() as client:
            resp = client.post(f"{self.backend_url}/api/v1/agent/chat", json=payload, timeout=60)
            resp.raise_for_status()
            data = resp.json()
        
        return {
            "answer": data.get("answer", ""), 
            "raw_tool_calls": data.get("tool_calls", []) 
        }

    # --- UPDATED TEST: TOOL USE ACCURACY ---
    def test_tool_use_accuracy(self):
        async def _run_test_logic():
            metric = ToolCallAccuracy(strict_order=False)
            # Only keep cases that declare which tool should be called
            subset = [d for d in self.dataset if d.get("expected_tool")]
            
            if not subset:
                self.skipTest("No data with 'expected_tool' found.")
                
            scores = []
            results_detail = []
            
            print(f"\n🚀 [TOOL ACCURACY] Starting evaluation on {min(len(subset), 20)} cases...")
            print(f"⚙️ Truncation Config: Answer @ {ANSWER_TRUNCATION_PERCENTAGE*100}%")

            for case in subset[:20]:
                case_id = case.get('id', 'unknown')
                try:
                    print(f"🔎 [{case_id}] Calling agent...")
                    res = self._call_agent(case["question"], case.get("ticker"))
                    
                    # 1. Dynamic Answer Truncation (Percentage-based)
                    ans = res.get("answer", "")
                    ans_len = len(ans)
                    ans_limit = int(ans_len * ANSWER_TRUNCATION_PERCENTAGE)
                    safe_answer = ans[:ans_limit] + "..." if ans_len > 0 else ""
                    
                    # 2. Extract and Deduplicate Tool Calls
                    raw_calls = res.get("raw_tool_calls", [])
                    unique_tool_names = list(set([tc.get("tool_name", "") for tc in raw_calls if tc.get("tool_name")]))
                    ai_tool_calls = [RagasToolCall(name=name, args={}) for name in unique_tool_names]
                    
                    # Check if tools were expected but none were called
                    expected_tool = case.get("expected_tool")
                    if not ai_tool_calls:
                        print(f"⚠️ [{case_id}] Agent made NO tool calls (Expected: {expected_tool})")
                    else:
                        print(f"🛠️ [{case_id}] Agent called: {unique_tool_names}")

                    # 3. Construct Ragas Message Objects
                    user_input = [
                        RagasHumanMessage(content=case["question"]),
                        RagasAIMessage(content=safe_answer, tool_calls=ai_tool_calls),
                    ]
                    
                    reference_tool_calls = [RagasToolCall(name=expected_tool, args={})] if expected_tool else []
                    
                    # 4. Calculate Score
                    result = await metric.ascore(
                        user_input=user_input,
                        reference_tool_calls=reference_tool_calls,
                    )
                    
                    score_val = result.value if hasattr(result, 'value') else float(result)
                    scores.append(score_val)
                    print(f"📊 [{case_id}] Score: {score_val:.4f}")

                    results_detail.append({
                        "id": case_id,
                        "question": case["question"],
                        "expected_tool": expected_tool,
                        "ai_tools": ", ".join(unique_tool_names),
                        "tool_call_accuracy": round(score_val, 4),
                        "passed": score_val >= 1.0 # Tool accuracy is usually binary (1.0 or 0.0)
                    })
                    
                except Exception as e:
                    print(f"❌ [{case_id}] Tool evaluation failed: {e}")
                    scores.append(0.0)
                    results_detail.append({
                        "id": case_id,
                        "question": case.get("question"),
                        "error": str(e),
                        "passed": False
                    })
            
            # Export to Excel
            if results_detail:
                df = pd.DataFrame(results_detail)
                try:
                    _save_to_excel(df, "ragas_tool_accuracy")
                except NameError:
                    pass
            
            mean_score = sum(scores) / len(scores) if scores else 0.0
            print(f"📈 [TOOL ACCURACY] Final Mean Score: {mean_score:.4f}")
            return mean_score

        # Bridging Sync test to Async logic using existing loop
        loop = asyncio.get_event_loop()
        score = loop.run_until_complete(_run_test_logic())
        
        self.assertGreaterEqual(score, TOOL_ACCURACY_THRESHOLD)

# 4. Execution Block
if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestToolUseAccuracy)
    unittest.TextTestRunner(verbosity=2).run(suite)

✅ Dataset loaded: 52 cases found.


test_tool_use_accuracy (__main__.TestToolUseAccuracy.test_tool_use_accuracy) ... 


🚀 [TOOL ACCURACY] Starting evaluation on 20 cases...
⚙️ Truncation Config: Answer @ 30.0%
🔎 [gd-001] Calling agent...
🛠️ [gd-001] Agent called: ['vector_store']
📊 [gd-001] Score: 1.0000
🔎 [gd-002] Calling agent...
🛠️ [gd-002] Agent called: ['get_fundamentals']
📊 [gd-002] Score: 1.0000
🔎 [gd-003] Calling agent...
🛠️ [gd-003] Agent called: ['vector_store']
📊 [gd-003] Score: 1.0000
🔎 [gd-004] Calling agent...
🛠️ [gd-004] Agent called: ['get_company_financials']
📊 [gd-004] Score: 1.0000
🔎 [gd-005] Calling agent...
🛠️ [gd-005] Agent called: ['vector_store']
📊 [gd-005] Score: 1.0000
🔎 [gd-006] Calling agent...
🛠️ [gd-006] Agent called: ['get_price_history']
📊 [gd-006] Score: 1.0000
🔎 [gd-007] Calling agent...
🛠️ [gd-007] Agent called: ['vector_store', 'get_fundamentals']
📊 [gd-007] Score: 0.0000
🔎 [gd-008] Calling agent...


/Users/juansebastianvargastorres/Desktop/UTS_subjects/Semester_2/NLP/Assignment2/finsightAI/.venv/lib/python3.13/site-packages/ragas/metrics/collections/tool_call_accuracy/metric.py:135: UserWarning: Length mismatch: predicted tool calls (2) vs reference tool calls (1). Only the first 1 tool calls will be compared.
  warnings.warn(


🛠️ [gd-008] Agent called: ['get_portfolio']
📊 [gd-008] Score: 1.0000
🔎 [gd-009] Calling agent...
🛠️ [gd-009] Agent called: ['vector_store']
📊 [gd-009] Score: 1.0000
🔎 [gd-010] Calling agent...
🛠️ [gd-010] Agent called: ['vector_store']
📊 [gd-010] Score: 1.0000
🔎 [gd-011] Calling agent...
🛠️ [gd-011] Agent called: ['get_fundamentals']
📊 [gd-011] Score: 1.0000
🔎 [gd-012] Calling agent...
🛠️ [gd-012] Agent called: ['vector_store']
📊 [gd-012] Score: 1.0000
🔎 [gd-013] Calling agent...
🛠️ [gd-013] Agent called: ['get_company_financials']
📊 [gd-013] Score: 1.0000
🔎 [gd-014] Calling agent...
🛠️ [gd-014] Agent called: ['get_company_financials']
📊 [gd-014] Score: 1.0000
🔎 [gd-015] Calling agent...
🛠️ [gd-015] Agent called: ['get_company_financials']
📊 [gd-015] Score: 1.0000
🔎 [gd-016] Calling agent...
🛠️ [gd-016] Agent called: ['get_company_financials']
📊 [gd-016] Score: 1.0000
🔎 [gd-017] Calling agent...
🛠️ [gd-017] Agent called: ['get_company_financials']
📊 [gd-017] Score: 1.0000
🔎 [gd-018] Ca

ok

----------------------------------------------------------------------
Ran 1 test in 150.609s

OK


🛠️ [gd-020] Agent called: ['get_price_history']
📊 [gd-020] Score: 1.0000
[INFO] Results saved to results/ragas_tool_accuracy_20260428.xlsx
📈 [TOOL ACCURACY] Final Mean Score: 0.9500


In [6]:
class TestResponseTime(unittest.TestCase): 
    
    @classmethod
    def setUpClass(cls):
        load_dotenv()
        cls.backend_url = os.getenv("BACKEND_URL", "http://localhost:8001")
        cls.dataset = _load_golden_dataset() 

    def _call_agent(self, question: str, ticker: str | None = None) -> dict:
        import httpx
        payload = {"message": f"[Ticker: {ticker}] {question}" if ticker else question}
        resp = httpx.post(f"{self.backend_url}/api/v1/agent/chat", json=payload, timeout=25)
        resp.raise_for_status()
        return resp.json()

    def test_agent_response_time(self):
        subset = self.dataset[:20]
        if not subset:
            self.skipTest("No data available to test response time.")
            
        response_times = []
        results_detail = []

        for case in subset[:20]:
            start_time = time.time()
            error_msg = None
            try:
                self._call_agent(case["question"], case.get("ticker"))
            except Exception as e:
                error_msg = str(e)
                print(f"[{case.get('id', 'unknown_id')}] Request failed during timing test: {e}")
            finally:
                elapsed_time = time.time() - start_time
                response_times.append(elapsed_time)
                print(f"[{case.get('id', 'unknown_id')}] Response Time: {elapsed_time:.2f}s")
                row = {
                    "id": case.get("id"),
                    "question": case["question"],
                    "response_time_s": round(elapsed_time, 4),
                    "passed": elapsed_time <= MAX_AVERAGE_RESPONSE_TIME,
                }
                if error_msg:
                    row["error"] = error_msg
                results_detail.append(row)

        if not response_times:
            self.fail("Could not measure any response times.")

        average_time = sum(response_times) / len(response_times)
        print(f"Average Agent Response Time: {average_time:.2f}s (Threshold: {MAX_AVERAGE_RESPONSE_TIME}s)")

        if results_detail:
            df = pd.DataFrame(results_detail)
            df.loc[len(df)] = {
                "id": "AVERAGE",
                "question": "",
                "response_time_s": round(average_time, 4),
                "passed": average_time <= MAX_AVERAGE_RESPONSE_TIME,
            }
            try:
                _save_to_excel(df, "response_time")
            except NameError:
                pass

        self.assertLessEqual(
            average_time,
            MAX_AVERAGE_RESPONSE_TIME,
            f"Average response time of {average_time:.2f}s exceeded the limit."
        )

if __name__ == '__main__':
    suite = unittest.TestLoader().loadTestsFromTestCase(TestResponseTime)
    unittest.TextTestRunner(verbosity=2).run(suite)

test_agent_response_time (__main__.TestResponseTime.test_agent_response_time) ... 

[gd-001] Response Time: 14.99s
[gd-002] Response Time: 6.25s
[gd-003] Response Time: 7.84s
[gd-004] Response Time: 4.53s
[gd-005] Response Time: 7.69s
[gd-006] Response Time: 7.77s
[gd-007] Response Time: 9.25s
[gd-008] Response Time: 7.38s
[gd-009] Response Time: 8.19s
[gd-010] Response Time: 9.41s
[gd-011] Response Time: 10.18s
[gd-012] Response Time: 6.64s
[gd-013] Response Time: 3.26s
[gd-014] Response Time: 3.47s
[gd-015] Response Time: 3.01s
[gd-016] Response Time: 3.48s
[gd-017] Response Time: 4.38s
[gd-018] Response Time: 8.89s
[gd-019] Response Time: 11.57s
[gd-020] Response Time: 8.60s
Average Agent Response Time: 7.34s (Threshold: 12.0s)


ok

----------------------------------------------------------------------
Ran 1 test in 147.136s

OK


[INFO] Results saved to results/response_time_20260428.xlsx
